In [ ]:
import os
import json

from openai import OpenAI

In [ ]:
API_KEY = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

In [ ]:
def evaluator_prompt(question_id, question, concepts, model_solving, target_answer):
    prompt = f"""You are an evaluator of mathematical solutions.
**Problem:**
{question}

**Concept Understanding:**
{concepts}

**Solution Process:**
{model_solving}

**Correct Answer:**
{target_answer}

**Your Task:**
Analyze ALL errors from these three categories:
1. **Concept Understanding Error**: Did the "Concept Understanding" reflect a misunderstanding of what the problem is asking? Is the concept description incorrect or incomplete?
2. **Variable/Equation Setup Error**: Did the "Solution Process" define variables incorrectly, write wrong equations, or establish incorrect relationships between variables?
3. **Calculation Error**: Did the "Solution Process" make an error in the calculation process (e.g., arithmetic error, algebra error, unit conversion error, etc...)?

**Response Format (JSON only):**
Return ONLY a valid JSON object with the following fields:
{{
  "problem_id": "{question_id}",
  "concept_error": true/false,
  "setup_error": true/false,
  "calculation_error": true/false
}}

**Decision  Rules:**
- Check for Concept Understanding Errors in the problem's Concept Understanding, and for Variable/Equation Setup or Calculation Errors in the Solution Process.
- Even if the Solution Process's final answer matches the target answer, check if the ENTIRE PROCESS was correct.
- concept_error = true if a Concept Understanding Error is present in the "Concept Understanding"; otherwise false.
- setup_error = true if a Variable/Equation Setup Error is present in the "Solution Process"; otherwise false.
- calculation_error = true if a Calculation Error is present in the "Solution Process"; otherwise false.
"""
    return prompt

In [ ]:
def convert_to_openai_batch_jsonl(data, output_file):
    # data(json) -> output_file(jsonl)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for example in data:
            
            # Create the analysis prompt
            analysis_prompt = evaluator_prompt(
                question_id=example['question_id'],
                question=example['question'],
                concepts=example['concepts'],
                model_solving=example['model_solving'],
                target_answer=example['target_answer'],
                
            )
            
            # Create OpenAI Batch API request format
            batch_request = {
                "custom_id": example['question_id'],
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-4o-mini-2024-07-18",  # or "gpt-4-turbo" or another model
                    "messages": [
                        {
                            "role": "system",
                            "content": "You are a superior math problem grader."
                        },
                        {
                            "role": "user",
                            "content": analysis_prompt
                        }
                    ],
                    "response_format": {"type": "json_object"},
                    "temperature": 0.0,
                    "max_tokens": 150
                }
            }
            
            # JSONL 
            f.write(json.dumps(batch_request) + '\n')
    
    #return output_file

In [ ]:
for i in range(1, 5):
    with open(f"subset_{i}_student_solving.json", "r", encoding="utf-8") as f:
        data = json.load(f)

    convert_to_openai_batch_jsonl(data, f"subset_{i}_batch.jsonl")

In [ ]:
## uploading the file
# after creating batch file, you must upload it so that you can reference it correctly when kicking off batches.
# upload josnl file using the Files API
subset_1_batch_file = client.files.create(
    file=open("subset_1_batch.jsonl", "rb"), # open the file in read-binary mode 
    purpose="batch" # specify the purpose of the file 
)

subset_2_batch_file = client.files.create(
    file=open("subset_2_batch.jsonl", "rb"), 
    purpose="batch" 
)

subset_3_batch_file = client.files.create(
    file=open("subset_3_batch.jsonl", "rb"), 
    purpose="batch" 
)

subset_4_batch_file = client.files.create(
    file=open("subset_4_batch.jsonl", "rb"), 
    purpose="batch" 
)

In [9]:
# get ID of the batch input file
subset_1_batch_file_id = subset_1_batch_file.id
subset_2_batch_file_id = subset_2_batch_file.id
subset_3_batch_file_id = subset_3_batch_file.id
subset_4_batch_file_id = subset_4_batch_file.id

In [10]:
# create a batch job using the batch input file 
subset_1_batch = client.batches.create(
    input_file_id=subset_1_batch_file_id,
    endpoint="/v1/chat/completions", # API endpoint to use for the batch
    completion_window="24h" # time for completion
)

In [ ]:
print(client.batches.retrieve(subset_1_batch.id))
print(client.batches.retrieve(subset_1_batch.id).status)

In [ ]:
content = client.files.content(client.batches.retrieve(subset_1_batch.id).output_file_id)
content.write_to_file("subset_1_batch_results.jsonl") 

In [ ]:
subset_2_batch = client.batches.create(
    input_file_id=subset_2_batch_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h" 
)

In [ ]:
print(client.batches.retrieve(subset_2_batch.id))
print(client.batches.retrieve(subset_2_batch.id).status)

In [ ]:
content = client.files.content(client.batches.retrieve(subset_2_batch.id).output_file_id)
content.write_to_file("subset_2_batch_results.jsonl")

In [ ]:
subset_3_batch = client.batches.create(
    input_file_id=subset_3_batch_file_id,
    endpoint="/v1/chat/completions", 
    completion_window="24h" 
)

In [ ]:
print(client.batches.retrieve(subset_3_batch.id))
print(client.batches.retrieve(subset_3_batch.id).status)

In [ ]:
content = client.files.content(client.batches.retrieve(subset_3_batch.id).output_file_id)
content.write_to_file("subset_3_batch_results.jsonl")

In [ ]:
subset_4_batch = client.batches.create(
    input_file_id=subset_4_batch_file_id,
    endpoint="/v1/chat/completions", 
    completion_window="24h" 
)

In [ ]:
print(client.batches.retrieve(subset_4_batch.id))
print(client.batches.retrieve(subset_4_batch.id).status)

In [38]:
content = client.files.content(client.batches.retrieve(subset_4_batch.id).output_file_id)
content.write_to_file("subset_4_batch_results.jsonl")

In [52]:
import jsonlines

with jsonlines.open("subset_4_batch_results.jsonl") as f:
    sampled_batch_results = list(f)

In [53]:
print(sampled_batch_results[9]["response"]["body"]["choices"][0]["message"]["content"])

{
  "problem_id": "q_5614",
  "concept_error": false,
  "setup_error": false,
  "calculation_error": true
}


In [54]:
results_dict = {}
for i in range(1, 5):
    with jsonlines.open(f"subset_{i}_batch_results.jsonl") as f:
        results_dict[f"subset_{i}"] = list(f)

In [55]:
def count_error_type(result_data):
    correct_answer_id, wrong_answer_id = [], []

    error_counts = {
        "co_only": 0,      # Concept Error만
        "se_only": 0,      # Setup Error만
        "ca_only": 0,      # Calculation Error만
        "co_se": 0,        # Concept + Setup
        "co_ca": 0,        # Concept + Calculation
        "se_ca": 0,        # Setup + Calculation
        "all_three": 0,    # 3개 다 틀린 경우
        }

    for i in range(len(result_data)):
        item = json.loads(result_data[i]["response"]["body"]["choices"][0]["message"]["content"])
        q_id = item["problem_id"]
        co = item["concept_error"]
        se = item["setup_error"]
        ca = item["calculation_error"]

        # 세 개가 모두 False인 경우 -> 정답
        if (not co) and (not se) and (not ca):
            correct_answer_id.append(q_id)
        
        else:
            # 하나라도 True인 경우 -> 오답 리스트에 추가하고 유형 분석
            wrong_answer_id.append(q_id)
        
            # 에러 유형별 카운트 
            if co and se and ca:
                error_counts["all_three"] += 1
            
            elif co and se and not ca:
                error_counts["co_se"] += 1
            
            elif co and not se and ca:
                error_counts["co_ca"] += 1
            
            elif not co and se and ca:
                error_counts["se_ca"] += 1
            
            elif co and not se and not ca:
                error_counts["co_only"] += 1
            
            elif not co and se and not ca:
                error_counts["se_only"] += 1
            
            elif not co and not se and ca:
                error_counts["ca_only"] += 1
        
            else:
                error_counts["others"] += 1
    
    assert (len(correct_answer_id) + len(wrong_answer_id )) == len(result_data)
    assert len(wrong_answer_id) == sum(error_counts.values())
    
    return correct_answer_id, wrong_answer_id, error_counts

In [ ]:
aggregated_data = {}

for i in range(1, 5):
    key = f"subset_{i}"
    
    co, wr, counts = count_error_type(results_dict[key])
    
    aggregated_data[key] = {
        "correct_ids": co,
        "wrong_ids": wr,
        "cnts": counts
    }

In [57]:
len(aggregated_data["subset_1"]["correct_ids"]), len(aggregated_data["subset_2"]["wrong_ids"])

(517, 1349)

In [ ]:
with open("aggregated_student_solving_grading_batch_results.json", "w", encoding="utf-8") as f:
    json.dump(aggregated_data, f, indent=2)

In [60]:
def aggregate_results(aggregated_data):
    total_cor_cnt = 0
    total_wro_cnt = 0
    
    total_error_counts = {
        'co_only': 0, 'se_only': 0, 'ca_only': 0,
        'co_se': 0, 'co_ca': 0, 'se_ca': 0, 'all_three': 0,
    }

    for i in range(1, len(aggregated_data) + 1):
        subset_key = f"subset_{i}"
        data = aggregated_data[subset_key]
        
        total_cor_cnt += len(data["correct_ids"])
        total_wro_cnt += len(data["wrong_ids"])
        
        subset_counts = data["cnts"] 
        
        for error_type, count in subset_counts.items():
            if error_type in total_error_counts:
                total_error_counts[error_type] += count

    # --- 데이터 확인 (디버깅용) ---
    print(f"정답: {total_cor_cnt}, 오답: {total_wro_cnt}")
    print(f"오답 유형: {total_error_counts}")

    return total_cor_cnt, total_wro_cnt, total_error_counts
    

In [61]:
total_cor_cnt, total_wro_cnt, total_error_counts= aggregate_results(aggregated_data)

정답: 2114, 오답: 5359
오답 유형: {'co_only': 163, 'se_only': 283, 'ca_only': 738, 'co_se': 380, 'co_ca': 274, 'se_ca': 1487, 'all_three': 2034}
